In [8]:
"""
stage5_lgbm.py -- Stage 5: LGBM hazard on Hawkes-derived continuous features.

RULES
  S5.1  Target and timing identical to Stage 2 hazard: predict is_target[t]
        from info <= t-1. Row order identical to Featurizer.build for the same
        date range (same _selected / non-warm path), so metas align.
  S5.2  Per (stream, class): bars-since-last-event with event <= t-1
        (censored at session start -> local_t + 1), and exponentially decayed
        event counters s_t = sum_{k>=1} a^k * ind_{t-k}, a = exp(-1/tau),
        tau in TAUS (bars). Both use events <= t-1 only.
  S5.3  age (S2.3 definition) and tod passed as raw continuous columns.
  S5.4  Information set = pivot grammar only, matching Stage 2. Any gain over
        the calibrated Hawkes measures nonlinearity/interactions, not new data.
  S5.5  Early stopping on a day-level chronological split: valid = train dates
        >= VALID_FROM, whole sessions only. No random row splits.
  S5.6  Isotonic calibration is fitted on the VALID fold predictions, not
        train: boosted-tree train predictions are systematically too extreme
        and would bias the calibrator.
  S5.7  Prediction parquet schema: bar_index, timestamp, is_target, p, p_cal.
        Drop-in for plot_day_hawkes.build_day_frame(pred=...).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

TAUS = (2.0, 6.0, 18.0)

LGBM_PARAMS = dict(
    objective="binary",
    metric="binary_logloss",
    learning_rate=0.05,
    num_leaves=127,
    min_data_in_leaf=1000,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    num_threads=16,
    verbosity=-1,
)
NUM_ROUNDS = 4000
EARLY_STOP = 200

# ---- CONFIG ----
FRAME = 6
BARS_PATH = f"data/stage-0/bars_{FRAME}s.parquet"
EVENTS_PATH = f"data/stage-0/events_{FRAME}s.parquet"
OUT_DIR = "data/stage-5"

TRAIN_START = None
TRAIN_END = "2024-12-31"
VALID_FROM = "2024-07-01"          # last 6 months of train = early-stop fold
TEST_FROM = "2025-01-01"
# ----------------
TOD_START_MIN = 570      # first session minute (9:30)
TOD_BIN_MIN = 30
N_TOD = 13               # bins covering the session 6.5hrs / 30 min

STREAMS = ["MNQ_D1", "MNQ_D2", "TICK_D1", "TICK_D2"]


In [2]:
def _make_classes():
    cls = []
    for s in STREAMS:
        cls.append((s, "opp"))
        cls.append((s, "conf"))
    cls.append(("MNQ_JMA_SELF", "all"))
    return cls

def _bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins

def _age_bins_from_edges(edges):
    b = _bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b

class Featurizer:
    def __init__(self, bars, events,
                 bin_edges=(1, 2, 3, 5, 8, 13, 21, 34),
                 age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89)):
        self.bin_edges = list(bin_edges)
        self.age_edges = list(age_edges)
        self.bins = _bins_from_edges(bin_edges)
        self.age_bins = _age_bins_from_edges(age_edges)
        self.classes = _make_classes()
        self.feature_names = (
            [f"{s}|{c}|lag{lo}_{hi}" for (s, c) in self.classes for (lo, hi) in self.bins]
            + [f"age|{lo}_{hi}" for (lo, hi) in self.age_bins]
            + [f"tod|{k}" for k in range(N_TOD)]
        )
        self.n_feat = len(self.feature_names)

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]   # S2.6
        evg = dict(tuple(ev.groupby("date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"])
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - TOD_START_MIN   # S2.10
            tod = np.clip(mins // TOD_BIN_MIN, 0, N_TOD - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == "MNQ_JMA_SELF":
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))              # P[i] = count pos < i
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _fill(self, S, t, out):
        n = S["n"]
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:                                         # S2.1, S2.2
                b = np.clip(t - lo + 1, 0, n)
                a = np.clip(t - hi, 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S2.3
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][t]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _fill_frozen(self, S, q, h, out):
        """S2.5: features at t = q+h using only events/targets with pos <= q."""
        n = S["n"]
        t = q + h
        cap = q + 1
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:
                b = np.clip(np.minimum(t - lo + 1, cap), 0, n)
                a = np.clip(np.minimum(t - hi, cap), 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = S["lt_incl"][q]
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][np.minimum(t, n - 1)]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S

    def build(self, date_from=None, date_to=None):
        sel = []
        total = 0
        for S in self._selected(date_from, date_to):
            t = np.nonzero(~S["warm"])[0]                                      # S2.7
            sel.append((S, t))
            total += len(t)
        X = np.zeros((total, self.n_feat), np.float32)
        y = np.zeros(total, np.int8)
        bar_index = np.zeros(total, np.int64)
        sess_id = np.zeros(total, np.int32)
        timestamp = np.zeros(total, "datetime64[ns]")
        ofs = 0
        for sid, (S, t) in enumerate(sel):
            m = len(t)
            self._fill(S, t, X[ofs:ofs + m])
            y[ofs:ofs + m] = S["tgt"][t]
            bar_index[ofs:ofs + m] = S["bar_index"][t]
            sess_id[ofs:ofs + m] = sid
            timestamp[ofs:ofs + m] = S["timestamp"][t]
            ofs += m
        meta = pd.DataFrame({"bar_index": bar_index, "timestamp": timestamp,
                             "sess_id": sess_id, "is_target": y.astype(bool)})
        return X, y, meta

In [3]:
# ---------------------------------------------------------------- features

def build_lgbm_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)            # 0/1 event indicator per bar
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))     # last event <= t-1   S5.2
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))          # events shifted to <= t-1
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)             # s_t = a*(s_{t-1} + ind_{t-1})
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S5.3
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))

    names = []
    for (s, c) in fz.classes:
        names.append(f"{s}|{c}|bsince")
        names += [f"{s}|{c}|ewm{tau:g}" for tau in TAUS]
    names += ["age", "tod"]
    return np.concatenate(blocks), names


def build_meta(fz, date_from=None, date_to=None):
    """Row-aligned with build_lgbm_features (S5.1)."""
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [4]:
# ---------------------------------------------------------------- train / eval

def train_lgbm(fz, train_start, train_end, valid_from,
               params=LGBM_PARAMS, num_rounds=NUM_ROUNDS, early_stop=EARLY_STOP):
    X, names = build_lgbm_features(fz, train_start, train_end)
    meta = build_meta(fz, train_start, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)

    va = (meta["date"] >= valid_from).to_numpy()                               # S5.5
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)

    booster = lgb.train(params, dtr, num_boost_round=num_rounds,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(early_stop, verbose=False),
                                   lgb.log_evaluation(200)])

    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])            # S5.6

    print(json.dumps(dict(
        n_train=int(tr.sum()), n_valid=int(va.sum()),
        best_iteration=int(booster.best_iteration),
        valid_logloss_raw=float(log_loss(y[va], p_va)),
        valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))), indent=2))

    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(15).to_string(index=False))

    return dict(booster=booster, iso=iso, feature_names=names,
                train_start=train_start, train_end=train_end,
                valid_from=valid_from, importance=imp)


def evaluate_lgbm(fz, model, start, end=None, stage2_anchor=0.30206):
    X, _ = build_lgbm_features(fz, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)

    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)

    ll_raw = log_loss(y, p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(
        n_rows=int(len(y)),
        holdout_logloss_raw=float(ll_raw),
        holdout_logloss_cal=float(ll_cal),
        holdout_logloss_const=float(ll_const),
        auc=float(roc_auc_score(y, p)),
        stage2_anchor=stage2_anchor,
        delta_vs_stage2=float(ll_cal - stage2_anchor)), indent=2))

    out = meta[["bar_index", "timestamp", "is_target"]].copy()                 # S5.7
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)

    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"),
                                  n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [9]:
# ---------------------------------------------------------------- usage
#from stage2_hawkes import Featurizer

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)
fz = Featurizer(bars, events)

model = train_lgbm(fz, TRAIN_START, TRAIN_END, VALID_FROM)
pred = evaluate_lgbm(fz, model, TEST_FROM)

joblib.dump({k: v for k, v in model.items() if k != "importance"},
            f"{OUT_DIR}/lgbm_model_{FRAME}s.joblib")
model["importance"].to_csv(f"{OUT_DIR}/lgbm_importance_{FRAME}s.csv", index=False)
pred.to_parquet(f"{OUT_DIR}/pred_lgbm_{FRAME}s.parquet", index=False)

[200]	valid's binary_logloss: 0.276275
[400]	valid's binary_logloss: 0.276086
[600]	valid's binary_logloss: 0.276174
{
  "n_train": 2462897,
  "n_valid": 495290,
  "best_iteration": 400,
  "valid_logloss_raw": 0.27608623508616886,
  "valid_logloss_cal": 0.2758316278398188
}
                feature          gain
      MNQ_D1|opp|bsince 437727.454641
MNQ_JMA_SELF|all|bsince 432158.809482
       MNQ_D1|conf|ewm2 360551.510287
     MNQ_D1|conf|bsince 324765.459621
  MNQ_JMA_SELF|all|ewm2 263695.699885
        MNQ_D1|opp|ewm2 245052.093621
        MNQ_D2|opp|ewm2  98756.601808
       MNQ_D2|conf|ewm2  60830.699232
       TICK_D1|opp|ewm2  53930.835129
      MNQ_D2|opp|bsince  45774.736203
      TICK_D1|conf|ewm2  37099.916022
       TICK_D2|opp|ewm2  33175.739425
      TICK_D2|conf|ewm2  24542.393894
  MNQ_JMA_SELF|all|ewm6  24510.842480
        MNQ_D1|opp|ewm6  23129.097926
{
  "n_rows": 1488181,
  "holdout_logloss_raw": 0.273890718297556,
  "holdout_logloss_cal": 0.2740234436386088,
  "ho